# 04 — Hardware Debugging

Analyze a QSSF recording from real or simulated hardware to identify faulty ancillas,
spatial error clustering, and temporal drift. This notebook is designed for
post-experiment diagnostic workflows.

**Requirements:** `pip install stabstream numpy matplotlib scipy`  
**Optional:** A `.qssf` recording file. Synthetic data is generated if none is provided.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict

# --- Configuration ---
RECORDING_PATH = None  # e.g. "../data/experiment_run.qssf"
DEM_PATH = None        # optional, for expected fire rates
N_ROUNDS = 2000
N_ANCILLAS = 24
P_PHYSICAL = 0.005
OUTLIER_SIGMA = 3.0    # flag ancillas > N sigma from mean

## 1. Load recording

In [ ]:
def load_recording(path: str) -> tuple[np.ndarray, list]:
    """Load (matrix, metadata_list) from QSSF. Returns (rounds, ancillas) matrix."""
    from stabstream import StabstreamStream
    rows, meta = [], []
    with StabstreamStream(path) as stream:
        for frame in stream:
            rows.append(frame.to_numpy_detector_events())
            meta.append({
                "frame_id": frame.frame_id,
                "round": frame.round,
                "timestamp_ns": frame.timestamp_ns,
                "observable_flips": frame.observable_flips,
            })
    return np.stack(rows), meta


def synthetic_recording(
    rounds: int, ancillas: int, p: float, seed: int = 42
) -> tuple[np.ndarray, list]:
    """Synthetic data with two injected outlier ancillas."""
    rng = np.random.default_rng(seed)
    matrix = (rng.random((rounds, ancillas)) < 2 * p).astype(bool)
    # Inject a leaky ancilla at index 7 (fires at 5× normal rate)
    matrix[:, 7] = rng.random(rounds) < min(10 * p, 0.9)
    # Inject a silent ancilla at index 15 (fires at 0.1× normal rate)
    matrix[:, 15] = rng.random(rounds) < 0.2 * p
    meta = [
        {"frame_id": i, "round": i, "timestamp_ns": i * 1100, "observable_flips": None}
        for i in range(rounds)
    ]
    return matrix, meta


if RECORDING_PATH and Path(RECORDING_PATH).exists():
    matrix, meta = load_recording(RECORDING_PATH)
    N_ROUNDS, N_ANCILLAS = matrix.shape
    print(f"Loaded {N_ROUNDS} rounds × {N_ANCILLAS} ancillas from {RECORDING_PATH}")
else:
    matrix, meta = synthetic_recording(N_ROUNDS, N_ANCILLAS, P_PHYSICAL)
    print(f"Synthetic data: {N_ROUNDS} rounds × {N_ANCILLAS} ancillas")
    print(f"  Injected leaky ancilla at index 7, silent ancilla at index 15")

## 2. Per-ancilla statistics

In [ ]:
freqs = matrix.mean(axis=0)  # (ancillas,)
expected = 2 * P_PHYSICAL

mean_f = freqs.mean()
std_f = freqs.std()
z_scores = (freqs - mean_f) / (std_f + 1e-12)

print(f"Global fire rate:  {mean_f:.4f}  (expected ~{expected:.4f})")
print(f"Std dev:           {std_f:.4f}")
print()
print(f"{'Ancilla':>8}  {'Freq':>8}  {'Z-score':>8}  {'Flag':>6}")
print("-" * 40)
outliers = []
for i, (f, z) in enumerate(zip(freqs, z_scores)):
    flag = ""
    if abs(z) > OUTLIER_SIGMA:
        flag = "🔴 HIGH" if z > 0 else "🔵 LOW"
        outliers.append(i)
    if abs(z) > 2.0 or i in [0, N_ANCILLAS // 2, N_ANCILLAS - 1]:
        print(f"  A{i:>4}    {f:>8.4f}  {z:>8.2f}  {flag}")

print(f"\nOutlier ancillas (|z| > {OUTLIER_SIGMA}): {outliers}")

## 3. Fire frequency visualization

In [ ]:
from stabstream.plot import plot_fire_frequency

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Bar chart
plot_fire_frequency(
    freqs, ax=axes[0],
    title="Per-Ancilla Fire Frequency",
    expected_rate=expected,
    outlier_threshold=OUTLIER_SIGMA,
)

# Z-score chart
colors = ["#d62728" if abs(z) > OUTLIER_SIGMA else "#aec7e8" for z in z_scores]
axes[1].bar(range(N_ANCILLAS), z_scores, color=colors, width=0.8)
for thresh in [OUTLIER_SIGMA, -OUTLIER_SIGMA]:
    axes[1].axhline(thresh, color="black", linestyle="--", linewidth=1, alpha=0.6)
axes[1].set_xlabel("Ancilla index")
axes[1].set_ylabel("Z-score")
axes[1].set_title("Fire Frequency Z-scores")
axes[1].set_xlim(-0.5, N_ANCILLAS - 0.5)

plt.tight_layout()
plt.savefig("fire_freq_debug.png", dpi=150)
plt.show()

## 4. Temporal drift analysis

Check if fire rates drift over the course of the experiment — a sign of qubit
parameter instability or environment fluctuations.

In [ ]:
window_size = max(50, N_ROUNDS // 20)
n_windows = N_ROUNDS // window_size
block_matrix = matrix[: n_windows * window_size].reshape(n_windows, window_size, N_ANCILLAS)

# Mean fire rate per time window per ancilla
block_freqs = block_matrix.mean(axis=1)  # (n_windows, ancillas)
window_centers = (np.arange(n_windows) + 0.5) * window_size

fig, ax = plt.subplots(figsize=(12, 5))
from stabstream.plot import plot_syndrome_heatmap
plot_syndrome_heatmap(
    block_freqs,
    ax=ax,
    title=f"Fire Rate Drift Over Time (window={window_size} rounds)",
    cmap="RdYlGn_r",
)
ax.set_xlabel("Ancilla index")
ax.set_ylabel(f"Time window ({window_size} rounds each)")
plt.tight_layout()
plt.savefig("temporal_drift.png", dpi=150)
plt.show()

# Compute per-ancilla drift (max - min across windows)
drift = block_freqs.max(axis=0) - block_freqs.min(axis=0)
worst = np.argsort(drift)[::-1][:5]
print("Top-5 drifting ancillas (max - min across time windows):")
for i in worst:
    print(f"  A{i}: drift = {drift[i]:.4f}  (z={z_scores[i]:.2f})")

## 5. Spatial correlation between ancillas

Correlated pairs suggest a shared error mechanism (e.g. a two-qubit gate is noisy).

In [ ]:
# Pearson correlation matrix between ancilla time series
corr = np.corrcoef(matrix.T.astype(float))  # (ancillas, ancillas)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1, interpolation="nearest")
plt.colorbar(im, ax=ax, label="Pearson r")
ax.set_title("Ancilla Fire Event Correlation Matrix")
ax.set_xlabel("Ancilla index")
ax.set_ylabel("Ancilla index")
if N_ANCILLAS <= 24:
    ax.set_xticks(range(N_ANCILLAS))
    ax.set_yticks(range(N_ANCILLAS))
    ax.set_xticklabels([f"A{i}" for i in range(N_ANCILLAS)], rotation=90, fontsize=7)
    ax.set_yticklabels([f"A{i}" for i in range(N_ANCILLAS)], fontsize=7)
plt.tight_layout()
plt.savefig("ancilla_correlation.png", dpi=150)
plt.show()

# Report highly correlated pairs (excluding diagonal)
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
pairs = np.argwhere(mask & (np.abs(corr) > 0.3))
if len(pairs):
    print("Highly correlated ancilla pairs (|r| > 0.3):")
    for (i, j) in pairs:
        print(f"  A{i} ↔ A{j}: r = {corr[i, j]:.3f}")
else:
    print("No strongly correlated ancilla pairs found (|r| ≤ 0.3).")

## 6. Timing analysis (if timestamps available)

In [ ]:
timestamps = np.array([m["timestamp_ns"] for m in meta], dtype=float)

if timestamps.max() > 0:
    intervals_ns = np.diff(timestamps)
    p50 = np.percentile(intervals_ns, 50)
    p99 = np.percentile(intervals_ns, 99)
    print(f"Frame interval: p50 = {p50:.0f} ns  p99 = {p99:.0f} ns")
    print(f"Estimated syndrome cycle: {p50/1e3:.2f} µs")

    fig, ax = plt.subplots(figsize=(8, 3))
    ax.hist(intervals_ns / 1e3, bins=50, color="#4363D8", edgecolor="none")
    ax.axvline(p50 / 1e3, color="red", linestyle="--",
               linewidth=1.5, label=f"p50 = {p50/1e3:.2f} µs")
    ax.set_xlabel("Frame interval (µs)")
    ax.set_ylabel("Count")
    ax.set_title("QSSF Frame Timing Distribution")
    ax.legend()
    plt.tight_layout()
    plt.savefig("frame_timing.png", dpi=150)
    plt.show()
else:
    print("No timestamp data available (synthetic recording).")

## 7. Diagnostic summary

In [ ]:
print("=" * 50)
print("HARDWARE DIAGNOSTIC SUMMARY")
print("=" * 50)
print(f"Total rounds analyzed:   {N_ROUNDS}")
print(f"Ancilla count:           {N_ANCILLAS}")
print(f"Global fire rate:        {mean_f:.4f}")
print(f"Expected ~2p:            {expected:.4f}")
delta = abs(mean_f - expected) / expected * 100
print(f"Deviation from expected: {delta:.1f}%")
print()

if outliers:
    high = [i for i in outliers if z_scores[i] > 0]
    low = [i for i in outliers if z_scores[i] < 0]
    if high:
        print(f"⚠  Overactive ancillas: {high}")
        print(f"   → Possible leakage, crosstalk, or noisy reset gate")
    if low:
        print(f"⚠  Underactive ancillas: {low}")
        print(f"   → Possible readout failure, decoupled qubit, or miscalibration")
else:
    print("✓  No outlier ancillas detected (all within 3σ)")

max_drift_ancilla = int(np.argmax(drift))
if drift[max_drift_ancilla] > 0.05:
    print(f"⚠  Temporal drift detected: A{max_drift_ancilla} drifted {drift[max_drift_ancilla]:.3f}")
    print(f"   → Consider recalibration mid-experiment")
else:
    print("✓  No significant temporal drift")

print()
print("Saved plots: syndrome_heatmap, fire_freq_debug, temporal_drift, ancilla_correlation")